In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, fixed


# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d, build_tissue
from render import render_image, to_rgb
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html, plot_surface_rgb_html, ortho, ortho_rgb, tau_cmap, NormalizeData
from optics import kryostat, psf_project, mask_collapse, detector
from parameter import (P, CellGeometry, MarkerPanel, PanelMarker, CellType, Detector,
                       dapi_marker,
                       BlobNoise, ClusterNoise, NetworkNoise, FibreNoise, SheetNoise,
                       Optics, TissueGeometry)
import config as cfg


%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED       = cfg.SEED
N_CAND     = cfg.N_CAND
SIZE       = cfg.SIZE
TILE       = cfg.TILE
K          = cfg.K
L_MIN, L   = cfg.L_MIN, cfg.L
UM_PER_VOX = cfg.UM_PER_VOX
Z_RATIO    = cfg.Z_RATIO
SPACING    = cfg.SPACING
VOL        = cfg.CELL_VOL
TISSUE_VOL = cfg.TISSUE_VOL
N_POOLS    = cfg.N_POOLS


GEOM = CellGeometry()
DETECTOR = Detector()
FRACTIONS = [0.7, 0.3]

# ONE POOL PER (marker, component) PAIR, so two markers using the same noise kind still draw
# independent frozen fields. 3 markers x 3 components = 9.
N_POOLS = 12

# 4) DRAW THE TAPE
tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile=TILE, n_cand=N_CAND, Pool=N_POOLS)
tape.draw3d(vol=VOL, n_cand=N_CAND, Pool=N_POOLS, l_min=L_MIN, L=L)

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, GEOM, size=VOL, spacing=SPACING, i=0, L=L, l_min=L_MIN)

In [ ]:
fig = plot_surface_xyz_inline(cell = cell, geom = GEOM)
fig = plot_surface_xyz_html(cell = cell, geom = GEOM,
                        out_path = Path("../renders").resolve())

In [ ]:
lab = cell["cell"].astype(np.int8) + cell["nuc"]        # 0 background, 1 cytoplasm, 2 nucleus
ortho(lab, cmap="viridis", title="mask:  0 background   1 cytoplasm   2 nucleus")
plt.show()

## 1.2) Cell Marker expression

### 1.2.2) Marker espression

In [ ]:
cm, vmin, vmax = tau_cmap(float(cell["tau"][cell["cell"]].max()))
ortho(cell["tau"], cmap=cm, vmin=vmin, vmax=vmax, mask=cell["cell"],
      title=r"$\tau$   (-1 nucleus centre,  0 nuclear envelope,  +1 plasma membrane)")
plt.show()

In [ ]:

PANEL = MarkerPanel(
    Markers = dict(
        DAPI = dapi_marker(),


        r = PanelMarker(name="r", fluorophore="APC", amp=2.0, polarity=0.2,
            noise_components=[
                ClusterNoise(w=.80, s=1.40, mu= .55, width=.45, sharp=5.0,
                             scale=.35, clust=2.00, fill=.22, soft=.25),
                FibreNoise  (w=.30, s=1.30, mu= .20, width=.70, sharp=4.0,
                             lam=.25, length=6.0),
                NetworkNoise(w=.40, s=1.30, mu= .50, width=.55, sharp=3.5,
                             scale=.80, coherence=.40),
            ], artifact_affinity=1.2),

        g = PanelMarker(name="g", fluorophore="FITC", amp=2.0, polarity=0.2,
            noise_components=[
                BlobNoise   (w=.50, s=1.50, mu= .10, width=.60, sharp=7.5,
                             scale=.45),
                SheetNoise  (w=.60, s=1.50, mu= .40, width=1.20, sharp=6.0,
                             lam=1.90, coherence=.35, length=6.0),
                NetworkNoise(w=.90, s=1.40, mu= .20, width=1.10, sharp=7.0,
                             scale=1.00, coherence=.60),
            ], artifact_affinity=0.2)
    )
)


IMG = (VOL[1], VOL[2], len(PANEL.names))
tape.drawSensor(shape=IMG)

# edge_softness=0 -> clipped EXACTLY at the plasma membrane. All the softness in the final
# image is made by psf_project and detector, not baked into the object.
# For this test, express at level 1 for every marker
SOLO = CellType(name="solo", Geometry=GEOM,
                Expression={k: P(0., 2., .05, 1.0, f"expr {k}") for k in PANEL.names})
img = render_image(tape, PANEL, cell, spacing=SPACING, geom=GEOM,
                   um_per_vox=UM_PER_VOX, edge_softness=0.0,
                   expression={k: SOLO.level(k) for k in PANEL.names})

rgb = to_rgb(cell["cell"], img)

In [ ]:
ortho_rgb(rgb, title="three markers, orthogonal slices through the cell")
plt.show()

In [ ]:
fig = plot_surface_rgb_html(cell = cell, elong = GEOM.ELONG.v,
                        polar_deg = GEOM.POLAR_DEG.v, azim_deg = GEOM.AZIM_DEG.v, roll_deg = GEOM.ROLL_DEG.v,
                        out_path = Path("../renders").resolve(), volume = rgb)

### 1.2.2) 2d Projection

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)


subs = {marker: kryostat(v, optics) for marker, v in img.items()}   # keep BOTH vol and z

img_psf = np.stack([psf_project(v, z, optics, PANEL.Markers[name].fluorophore)
                    for name, (v, z) in subs.items()], -1)

In [ ]:
sub, sub_z = kryostat(cell["cell"], optics)
sub_n, sub_n_z = kryostat(cell["nuc"], optics)


plt.imshow(NormalizeData(img_psf))
plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
# Pre-normalize
img_psf_norm = (img_psf - np.min(img_psf)) / (np.max(img_psf) - np.min(img_psf))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

### 1.2.2) 2d Projection + Detector Model

In [ ]:
markers = list(subs.keys())

img_adu = detector(img_psf, PANEL, DETECTOR, optics, tape, markers)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(NormalizeData(img_psf), origin="lower")
ax[0].set_title("clean projection (object x PSF)")
ax[1].imshow(NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0)), origin="lower")
ax[1].set_title("detector frame (AF + shot + read)")
ax[2].imshow(NormalizeData(img_adu[..., 0]), cmap="magma", origin="lower")
ax[2].set_title(f"channel 0 ({markers[0]}, {PANEL.Markers[markers[0]].fluorophore}) in ADU")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Pre-normalize
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

# 2) Synthetic tissue generation

In [ ]:
# For now, just two predesigned cell types
def _marker(name, dye, comps, amp=2.0):
    return PanelMarker(name=name, fluorophore=dye, amp=amp, polarity=0.0,
                       noise_components=comps)

PANEL = MarkerPanel(Markers=dict(
    DAPI=dapi_marker(),
    r=_marker("r", "APC", [ClusterNoise(w=.8, s=1.4, mu=.55, width=.45, sharp=5.,
                                        scale=.35, clust=2.0, fill=.22, soft=.25),
                           FibreNoise(w=.3, s=1.3, mu=.2, width=.7, sharp=4.,
                                      lam=.25, length=6.)], amp=2.4),
    g=_marker("g", "FITC", [BlobNoise(w=.5, s=1.5, mu=.1, width=.6, sharp=7.5, scale=.45),
                            NetworkNoise(w=.9, s=1.4, mu=.2, width=1.1, sharp=7.,
                                         scale=1.0, coherence=.6)], amp=1.2)
))


TISSUE_CELL = CellGeometry(RADIUS=8.0, ROUGH=0.15, ELONG=1.4, NUC_FRAC=0.40, RIM=0.0,
                           NUC_OFFSET=0.8)

def _expr(**levels):
    return {k: P(0., 2., .05, float(v), f"expr {k}",
                 comment="expression level; 0 = negative, 1 = the panel's nominal brightness")
            for k, v in levels.items()}

# stroma is r-high / b-mid and g-negative; tumour is the other way round. Same textures --
# only the levels differ, which is exactly how a real panel separates cell types.
CELLTYPES = {
    "stroma": CellType(name="stroma", Color="red", Geometry=TISSUE_CELL,
                       Expression=_expr(DAPI=1.0, r=1.2, b=0.9)),
    "tumour": CellType(name="tumour", Color="blue", Geometry=TISSUE_CELL,
                       Expression=_expr(DAPI=1.0, g=1.4, b=0.6, r=0.15)),
}
FRACTIONS = [0.7, 0.3]
T_POOLS = PANEL.n_pools()
tape.drawTissue(shape=TISSUE_VOL, n_cand=N_CAND, Pool=T_POOLS)

TG = TissueGeometry()

In [ ]:
t_img, labels, nuc_labels, tau_img, types, info = build_tissue(
    tape=tape, TG=TG, shape=TISSUE_VOL, base_geom=TISSUE_CELL, spacing=SPACING,
    Panel=PANEL, CellTypes=CELLTYPES, Fractions=FRACTIONS, um_per_vox=UM_PER_VOX,
    L=L, l_min=L_MIN)
t_rgb = to_rgb(labels > 0, t_img)

In [ ]:
cmap = {name: ct.Color for name, ct in CELLTYPES.items()}
# PLot tissue mask
mask = info['support'][0].astype(int)
cv = np.array([info["centres_vox"][n] for n in info["labels_present"]])
for i in range(info['support'].shape[0]-1):
    mask += info['support'][i].astype(int)
    plt.scatter(cv[i,0], cv[i,1], c = cmap[types[i+1]])
plt.imshow(mask)

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)
tape.drawSensor(shape=(160, 160, 3))

subs = {marker: kryostat(v, optics) for marker, v in t_img.items()}   # keep BOTH vol and z

# We can just use some cell type here as they all use the same channel names
t_psf = np.stack([psf_project(v, z, optics, PANEL.Markers[name].fluorophore)
                  for name, (v, z) in subs.items()], -1)
t_markers = list(subs)
img_adu = detector(t_psf, PANEL, DETECTOR, optics, tape, t_markers)

In [ ]:
sub, sub_z = kryostat(labels, optics)

# plt.imshow(np.sum(sub, axis=0))
plt.imshow(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[1])# , cmap = "Purples_r"
plt.axis('off')
plt.show()
plt.imshow(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0])# , cmap = "Purples"
plt.axis('off')
plt.show()

In [ ]:
sub, sub_z = kryostat(labels, optics)
sub_n, sub_n_z = kryostat(nuc_labels, optics)


plt.imshow(NormalizeData(img_adu))
# plt.contour(sub[0], colors="red" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green
# " , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99, keep_largest=False)[0], colors="yellow" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

# `lbl`/`lbl_z` (and the nucleus pair) are passed through interact's `fixed(...)` below, so the
# callback binds the tissue label slab captured when THIS cell runs. Without that it would read
# the module-level `sub`, which Section 3's load cell reassigns to a 4D frame batch -- feeding a
# 4D array into mask_collapse then crashes fftconvolve with a dimensionality mismatch.
def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0,
                 show_masks=True,
                 lbl=None, lbl_z=None, lbl_n=None, lbl_n_z=None):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    
    # Checkbox logic
    if show_masks:
        # plt.contour(lbl.any(0), colors="red" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
        plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
        # plt.contour(lbl.any(0), colors="red" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
        
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0), # Added missing comma here
         show_masks=Checkbox(value=True, description='Show Masks'),
         # bind the tissue slab now, so a later `sub` reassignment can't reach this callback
         lbl=fixed(sub), lbl_z=fixed(sub_z),
         lbl_n=fixed(sub_n), lbl_n_z=fixed(sub_n_z)
)
print("done")

## 2.1) Introducing Artifacts

In [ ]:
from parameter import Artifacts, ArtifactMap, Fussel, Aggregate, ArtifactMask

In [ ]:
AR = Artifacts(
    Map=ArtifactMap(MIN_DIST=12.0, COVER=0.60),
    Fussel=Fussel(N=1, WIDTH_UM=2.5, WOBBLE_UM=7.0),
    Aggregate=Aggregate(N=8, DIAM_UM=1.2, GAIN=120.0),
    Mask=ArtifactMask(DILATE_PX=2.0),
)

tape.drawArtifacts(shape=TISSUE_VOL, **cfg.ARTIFACTS)

In [ ]:
from scene import support_mask, thin, vox_to_world, body_frame, make_boundary, patch_grid, centred_grid_3d, TapeDict
from render import render_marker
import dataclasses
from scipy import ndimage as ndi

def artifact_sites(tape, AR, shape, n, tissue_support, skip = 0):
    nz, ny, nx = shape
    sup = support_mask(tape["art_support3"], AR.SUPPORT_SCALE.v, AR.SUPPORT_SCALE_Z.v,
                       AR.COVER.v)
    if tissue_support is not None:
        sup = sup & np.asarray(tissue_support, bool)
    col = sup.any(0)

    xy = tape["art_xy"]
    keep = thin(xy, tape["art_order"], AR.MIN_DIST.v)
    cx, cy = xy[keep, 0], xy[keep, 1]
    keep = keep[col[np.clip(cy.astype(int), 0, ny - 1), 
                    np.clip(cx.astype(int), 0, nx - 1)]]

    take = keep[skip:skip + max(int(round(n)), 0)]
    return xy[take], take

def marker_affinity(panel, names, dapi_name=cfg.DAPI_NAME):
    """Which markers artifacts contaminate, and how strongly. DAPI is always exactly 0."""
    out = {nm: float(panel.Markers[nm].artifact_affinity.v) for nm in names}
    out[dapi_name] = 0.0
    return out

def _perimeter_point(s, ny, nx, pad):
    """s in [0,1) -> a point on the frame boundary, pushed `pad` voxels OUTSIDE it.
    """
    H, W = ny + 2.0 * pad, nx + 2.0 * pad
    t = (float(s) % 1.0) * 2.0 * (H + W)
    if t < W:
        # top
        return (-pad, t - pad)
    t -= W
    if t < H:
        # right
        return (t - pad, nx + pad)
    t -= H
    if t < W:
        # bottom
        return (ny + pad, nx + pad - t)
    t -= W
    # left
    return (ny + pad - t, -pad)

def _mode_envelope(n_mode, beta, target=2.0):
    """m^-beta, renormalised so the wobble's RMS is 1 whatever beta or n_mode are.
    """
    m = np.arange(1, n_mode + 1, dtype=float)
    w = m ** -float(beta)
    return w * np.sqrt(target / max(float((w ** 2).sum()), 1e-30))

def _slab_bounds(shape, opt, margin_vox=1.0):
    """The voxel range kryostat will actually keep, minus a margin."""
    nz = shape[0]
    c = (nz - 1) / 2.0
    lim = 0.5 * float(opt.section_um) / float(opt.um_per_pz)
    lo, hi = c - lim + margin_vox, c + lim - margin_vox
    # An object thicker than the section cannot fit inside it at all; centre it and let the cut
    # take the top and bottom off symmetrically
    return (c, c) if lo > hi else (lo, hi)


def _z_centre_vox(z_frac, shape, opt, margin_vox=1.0):
    """Axial centre in VOXEL index, from a signed fraction of the half-section.
    """
    nz = shape[0]
    z_um = -float(z_frac) * 0.5 * float(opt.section_um)
    z_vox = (nz - 1) / 2.0 + z_um / float(opt.um_per_pz)
    lo, hi = _slab_bounds(shape, opt, margin_vox)
    return float(np.clip(z_vox, lo, hi))


def fibre_centreline(tape, j, F, shape, opt, um_per_vox, n_t=512, pad=None, width_vox=1.0):
    """The fibre's path: a chord across the frame, bent by a few frozen modes.
    Returns pts (n_t, 3) in (z, y, x) voxel coordinates.
    """
    nz, ny, nx = shape
    pad = float(np.ceil(2.0 * width_vox + 4.0)) if pad is None else pad
    sA = tape["u_fib_end"][j, 0]
    # Move the exit point sB between 0.3 - 0.7 of a full lap further 
    sB = sA + 0.30 + 0.40 * tape["u_fib_end"][j, 1]
    
    # Compute start and endpoint on the perimeter
    ay, ax = _perimeter_point(sA, ny, nx, pad)
    by, bx = _perimeter_point(sB, ny, nx, pad)

    # Compute the straight connecting line between the points
    dy, dx = by - ay, bx - ax
    norm = np.hypot(dy, dx) + 1e-12
    # in-plane normal to the chord
    ny_hat, nx_hat = -dx / norm, dy / norm            

    # Which fraction of the chord is actually drawn.
    # Threshould drawn value with learnable param
    u = tape["u_fib_span"][j]
    p_in, trim = float(F.P_END_INSIDE.v), float(F.END_TRIM.v)
    t_min = min(1.6 * pad / norm, 0.5 * trim)
    t0 = t_min + (trim - t_min) * u[2] if u[0] < p_in else 0.0
    t1 = 1.0 - (t_min + (trim - t_min) * u[3]) if u[1] < p_in else 1.0
    t = np.linspace(t0, max(t1, t0 + 0.05), int(n_t))
    yy = ay + t * (by - ay)
    xx = ax + t * (bx - ax)

    # 1D version of the same spectral trick in make_boundary to generate unit RMS
    n_mode = tape["z_fib_lat"].shape[1]
    w = _mode_envelope(n_mode, F.BETA.v)
    basis = np.sin(np.pi * np.arange(1, n_mode + 1)[None, :] * t[:, None])
    
    plt.scatter([ax, bx], [ay, by])
    plt.plot([ax, bx], [ay, by])
    plt.plot(xx, yy)

    lat = (basis * (w * tape["z_fib_lat"][j, :n_mode])).sum(1)
    lat_vox = float(F.WOBBLE_UM.v) / float(um_per_vox)
    yy = yy + lat_vox * lat * ny_hat
    xx = xx + lat_vox * lat * nx_hat

    plt.scatter(xx, yy)
    plt.show()
    # Add z-Axis wobble, but make sure it stays below cover slip
    z0 = _z_centre_vox(F.Z_FRAC.v, shape, opt, margin_vox=width_vox + 1.0)
    zlo, zhi = _slab_bounds(shape, opt, margin_vox=width_vox + 1.0)
    room = max(min(z0 - zlo, zhi - z0), 0.0)
    ax_raw = (basis * (w * tape["z_fib_ax"][j, :n_mode])).sum(1)
    zz = z0 + float(F.Z_WOBBLE_FRAC.v) * np.tanh(ax_raw) * room
    plt.scatter(np.arange(len(zz)), zz)
    plt.show()
    return np.stack([zz, yy, xx], -1)

def make_fussel(tape, F, j, shape, opt, cfg, um_per_vox, n_t, spacing):
    nz, ny, nx = shape
    # get radius of fussel
    r_um = 0.5 * float(F.WIDTH_UM.v) * np.exp(float(F.WIDTH_SIGMA.v) * tape["z_art_size"][j])
    # prevent R going -> 0
    R = max(float(r_um) / float(um_per_vox), 0.6)
    
    pts = fibre_centreline(tape, j, F, shape, opt, um_per_vox, n_t=n_t, width_vox=R)
    
    # Crop in z before the distance transform.
    zl = int(max(0, np.floor(pts[:, 0].min() - 2.0 * R - 2)))
    zh = int(min(nz, np.ceil(pts[:, 0].max() + 2.0 * R + 3)))
    sub = (slice(zl, zh), slice(0, ny), slice(0, nx))
    sz = (zh - zl, ny, nx)
    
    # rasterise densely enough to stay connected
    length = float(np.sum(np.linalg.norm(np.diff(pts[:, 1:], axis=0), axis=1)))
    n_s = max(int(np.ceil(2.5 * length)), n_t)
    ti = np.linspace(0.0, 1.0, n_s)
    ps = np.stack(
        [np.interp(ti, np.linspace(0, 1, n_t), pts[:, k]) for k in range(3)], -1
    )
    iz = np.rint(ps[:, 0]).astype(int) - zl
    iy = np.rint(ps[:, 1]).astype(int)
    ix = np.rint(ps[:, 2]).astype(int)
    ok = (iz >= 0) & (iz < sz[0]) & (iy >= 0) & (iy < ny) & (ix >= 0) & (ix < nx)
    if not ok.any():
        # the whole fibre missed the volume -- legal (it can wander outside), just render nothing
        return None
    
    # initialize seed map with false for the fussel voxels for distance transform
    seed = np.ones(sz, bool)
    seed[iz[ok], iy[ok], ix[ok]] = False
    # Returns the distance to nearest centreline and the nearest pixel on centreline
    dist, inds = ndi.distance_transform_edt(seed, sampling=spacing, return_indices=True)
    mask = dist <= R
    # get normalised radial coordinate: 0 on the centreline, 1 exactly at the surface,
    # clip due to large tau size of tile spanning fibres
    d = np.minimum(dist / R, cfg._TAU_CLIP)
    
    # Generate PHI
    # Initialize plain voxel index meshgrid
    zz, yy, xx = np.meshgrid(
        *[np.arange(s, dtype=np.float32) for s in sz], indexing="ij"
    )
    # Compute displacement from nearest centreline by subtracting inds
    sz_, sy_, sx_ = spacing
    vec = np.stack(
        [(xx - inds[2]) * sx_, (yy - inds[1]) * sy_, (zz - inds[0]) * sz_], -1
    )
    phi = vec / np.maximum(np.linalg.norm(vec, axis=-1, keepdims=True), 1e-6)

    geom = {
        "cell": mask,
        "tau": (2.0 * d - 1.0).astype(np.float32),
        "d": d.astype(np.float32),
        "phi": phi.astype(np.float32),
    }
    return {"geom": geom, "sl": sub, "pts": pts, "radius_vox": R, "length_vox": length}
    
def make_aggregates(tape, j, G, xyz, shape, spacing, um_per_vox, grid, L=4, l_min=2):
    r_um = 0.5 * float(G.DIAM_UM.v) * np.exp(float(G.DIAM_SIGMA.v) * tape["z_art_size"][j])
    R = max(float(r_um) / float(um_per_vox), 0.5)
    reach = int(np.ceil(R * np.exp(3.0 * float(G.ROUGH.v)))) + 2
    sl = patch_grid(xyz, reach, shape)
    if any(s.stop - s.start <= 0 for s in sl):
        return None

    centre = vox_to_world(xyz, shape, spacing)
    rho, u = body_frame(grid[sl], rot=None, centre=centre, elong=1.0)
    r_fn = make_boundary(coeff=tape["art_sh"][j], R=R, kappa=float(G.ROUGH.v), beta=2.0,
                        L=L, l_min=l_min)
    d = rho / np.maximum(r_fn(u), 1e-6)
    mask = d <= 1.0
    d = np.minimum(d, cfg._TAU_CLIP)          # see the note in make_fussel
    
    return {"geom":{
                "cell": mask, 
                "tau": (2.0 * d - 1.0).astype(np.float32),
                "d": d.astype(np.float32), 
                "phi": u.astype(np.float32)
            }, 
            "sl": sl, 
            "radius_vox": R}

def to_artifact_marker(marker, mu, width, sharp):
    """The panel marker's TEXTURE, relocalised onto the artifact's own tau."""
    # comps = [dataclasses.replace(c, mu=mu, width=width, sharp=sharp)
    #          for c in marker.noise_components]
    comps = []
    return dataclasses.replace(marker, noise_components=comps)

def build_artifacts(vols, tape, cfg, AR, opt, shape, panel, um_per_vox, spacing, geom = None, tissue_support= None, 
                    pool_bank=None ,L=4, l_min=2, n_t=512
                #     , TG, shape, base_geom, spacing, Panel, CellTypes, Fractions,
                #  um_per_vox, L=4, l_min=2, rule=None, pool_bank=None
                 ):
    """Tissue in three phases: geometry, then type assignment, then appearance.
    n_t is the fussels centreline's own sampling resolution (points along the curve)"""
    names = list(panel.Markers)
    
    # get the locations for fussels / Aggregats
    F, G = AR.Fussel, AR.Aggregate
    n_fib = max(int(round(F.N.v)), 0)
    n_agg = max(int(round(G.N.v)), 0)
    
    _, fib_rows = artifact_sites(tape, AR.Map, shape, n_fib, tissue_support)
    agg_xy, agg_rows = artifact_sites(tape, AR.Map, shape, n_agg, tissue_support,
                                      skip=len(fib_rows))
    
    nid = cfg.ARTIFACT_ID0
    table, affinities = [], {}
    grid = centred_grid_3d(shape, spacing)
    base_geom = CellGeometry() if geom is None else geom
    art_labels = np.zeros(shape, np.int32)
    
    
    aff = marker_affinity(panel, names)
    
    # FUSSEL generation
    for j in fib_rows:
        fb = make_fussel(tape=tape, F=F, j=int(j), shape = shape, opt = opt, cfg = cfg, um_per_vox=um_per_vox, n_t = n_t, spacing = spacing)
        if not fb["geom"]["cell"].any():
            continue
        
        # Marker expression
        affinities[nid] = aff
        gain = float(F.GAIN.v) * np.exp(float(F.GAIN_SIGMA.v) * tape["z_art_gain"][int(j)])
        sl, gm = fb["sl"], fb["geom"]
        ptape = TapeDict(
            texture_noise=tape["texture_noise"][(slice(None),) + sl],
            gate_noise=tape["gate_noise"][(slice(None),) + sl],
        )
        off = 0
        for name in names:
            m = panel.Markers[name]
            # Compute expression based on affinity
            lvl = aff[name] * gain
            if lvl > 1e-3:
                img = render_marker(tape=ptape, marker=to_artifact_marker(m, F.MU, F.WIDTH,
                                                                        F.SHARP),
                                    cell=gm, spacing=spacing, geom=base_geom,
                                    um_per_vox=um_per_vox, edge_softness=0.0,
                                    pool_offset=off, pool_bank=pool_bank)
                vols[name][sl] += np.float32(lvl) * img
            off += len(m.noise_components)
        art_labels[sl] = np.where(gm["cell"], nid, art_labels[sl])
        table.append(dict(id=nid, kind="fussel",
                        diam_um=2.0 * fb["radius_vox"] * um_per_vox, 
                        n_markers=int(sum(v > 1e-3 for v in aff.values())),
                        gain=gain))
        nid += 1
        
    ab = [n for n in names if n != cfg.DAPI_NAME]
    for k, j in enumerate(agg_rows):
        if not ab:
            # stop if there are no markers expressed
            break
        xyz = np.array([agg_xy[k][0], agg_xy[k][1], tape["u_art_z"][int(j)] * (shape[0] - 1)])
        ag = make_aggregates(tape, int(j), G, xyz, shape, spacing, um_per_vox, grid,
                            L=L, l_min=l_min)
        
        if not ag["geom"]["cell"].any(): 
            continue
        
        # Marker expression
        ch = ab[min(int(tape["u_art_ch"][int(j)] * len(ab)), len(ab) - 1)]
        gain = float(G.GAIN.v) * np.exp(float(G.GAIN_SIGMA.v) * tape["z_art_gain"][int(j)])
        sl, gm = ag["sl"], ag["geom"]
        ptape = TapeDict(texture_noise=tape["texture_noise"][(slice(None),) + sl],
                         gate_noise=tape["gate_noise"][(slice(None),) + sl])
        off = 0
        for name in names:
            m = panel.Markers[name]
            # one antibody precipitated, so it shows in ITS channel. SPILL relaxes that.
            lvl = gain if name == ch else (gain * float(G.SPILL.v) if name in ab else 0.0)
            if lvl > 1e-3:
                img = render_marker(tape=ptape, marker=to_artifact_marker(m, G.MU, G.WIDTH,
                                                                       G.SHARP),
                                    cell=gm, spacing=spacing, geom=base_geom,
                                    um_per_vox=um_per_vox, edge_softness=0.0,
                                    pool_offset=off, pool_bank=pool_bank)
                vols[name][sl] += np.float32(lvl) * img
            off += len(m.noise_components)
        art_labels[sl] = np.where(gm["cell"], nid, art_labels[sl])
        table.append(dict(id=nid, kind="aggregate", channel=ch,
                        diam_um=2.0 * ag["radius_vox"] * um_per_vox, gain=gain))
        nid += 1
        
    short = (max(n_fib - len(fib_rows), 0), max(n_agg - len(agg_rows), 0))
    return {
        "labels": art_labels,
        "table": table,
        "affinity": affinities,
        "n_fussel": sum(r["kind"] == "fussel" for r in table),
        "n_aggregate": sum(r["kind"] == "aggregate" for r in table),
        "shortfall": {"fussel": short[0], "aggregate": short[1]},
    }
    
t_img_art = {k: v.copy() for k, v in t_img.items()}

art = build_artifacts(vols = t_img_art, tape = tape, cfg = cfg, AR = AR, opt = optics, shape = TISSUE_VOL, panel = PANEL, um_per_vox = UM_PER_VOX, spacing = SPACING, geom=TISSUE_CELL, tissue_support=info["support"], L=L, l_min=L_MIN)
def render_adu(vols):
    """The unchanged tissue path: kryostat -> psf_project -> detector."""
    ss = {m: kryostat(v, optics) for m, v in vols.items()}
    psf = np.stack([psf_project(v, z, optics, PANEL.Markers[n].fluorophore)
                    for n, (v, z) in ss.items()], -1)
    return detector(psf, PANEL, DETECTOR, optics, tape, list(ss))


fig, ax = plt.subplots(1, 2, figsize = (16, 8))
adu_art = render_adu(t_img_art)
adu = render_adu(t_img)
sub_art, z_art = kryostat(art["labels"], optics)
ax[0].imshow(NormalizeData(adu_art))
ax[0].contour(mask_collapse(sub_art, z_art, optics, mask_pct=0.90)[0], colors="violet" , origin="lower", alpha=1.)
ax[1].imshow(NormalizeData(adu))
plt.show()


In [ ]:
img_psf_norm = NormalizeData(np.maximum(adu_art - DETECTOR.OFFSET_ADU.v, 0))

# `lbl`/`lbl_z` (and the nucleus pair) are passed through interact's `fixed(...)` below, so the
# callback binds the tissue label slab captured when THIS cell runs. Without that it would read
# the module-level `sub`, which Section 3's load cell reassigns to a 4D frame batch -- feeding a
# 4D array into mask_collapse then crashes fftconvolve with a dimensionality mismatch.
def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0,
                 show_masks=True, show_art_masks=True,
                 lbl=None, lbl_z=None, sub_art=None, z_art=None):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    
    # Checkbox logic
    if show_masks:
        plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        
    if show_art_masks:
        plt.contour(mask_collapse(sub_art, z_art, optics, mask_pct=0.90)[0], colors="violet" , origin="lower", alpha=.55)
        
        
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0), # Added missing comma here
         show_masks=Checkbox(value=True, description='Show Masks'),
         show_art_masks=Checkbox(value=True, description='Show Artifact Masks'),
         # bind the tissue slab now, so a later `sub` reassignment can't reach this callback
         lbl=fixed(sub), lbl_z=fixed(sub_z),
         sub_art=fixed(sub_art), z_art=fixed(z_art)
)
print("done")

# 3) Inspect a generated dataset

In [ ]:
# ============================================================================================
# LOAD.  Point DATA at the --out STEM used when generating (no .npy / .meta.npz suffix).
# ============================================================================================
DATA = Path("../data/example_small")

_stem = str(DATA.resolve())
if not os.path.exists(_stem + ".npy"):
    raise FileNotFoundError(
        f"no dataset at {_stem}.npy -- generate one first:\n"
        f"    python src/gen_dataset.py make --mode tissue --n 500 --workers 8 --out {_stem}")

meta = np.load(_stem + ".meta.npz", allow_pickle=True)
# mmap_mode="r" is the whole point: the frames stay on disk and only the ones indexed below
# are read. np.load without it would pull the entire set into RAM.
imgs = np.load(_stem + ".npy", mmap_mode="r")

theta = meta["theta"]
paths = [str(p) for p in meta["paths"]]
n, H, W, C = imgs.shape
# `names`/`dyes` are per channel in image order; older files predate them
names = [str(x) for x in meta["names"]] if "names" in meta.files else \
        [cfg.DAPI_NAME] + [f"ch{k}" for k in range(1, C)]
print(meta["dyes"])
dyes = [str(x) for x in meta["dyes"]] if "dyes" in meta.files else None

print(f"{_stem}")
print(f"  {n} frames, {H}x{W} px, {C} channels, {imgs.dtype} seed={meta['seed']}")
print(f"  {os.path.getsize(_stem + '.npy') / 1e9:.3f} GB on disk, memmapped -- resident cost "
      f"is one frame ({H * W * C * 2 / 1e6:.2f} MB), not the file")
print(f"  theta {theta.shape}: {len(paths)} fitted parameters, normalised to [0, 1]")
print("  channels: " + ", ".join(f"{k}:{nm}" + (f"/{dyes[k]}" if dyes else "")
                                 for k, nm in enumerate(names)))

# Statistics from a SUBSAMPLE. Touching every frame of a large set would read the whole file
# back off disk and undo the memmap; 64 frames is plenty to see the intensity regime.
# NB: this is a 4D frame batch -- do NOT call it `sub`, which the tissue widget above binds
# to a 3D label slab.
rng = np.random.default_rng(0)
pick = np.sort(rng.choice(n, size=min(n, 64), replace=False))
sample = np.asarray(imgs[pick], np.float32)          # only these rows leave the disk

print(f"\nper-channel ADU over {len(pick)} sampled frames "
      f"(black level = DETECTOR OFFSET_ADU = {cfg.DETECTOR['OFFSET_ADU']:.0f})")
print(f"  {'channel':>10}  {'median':>8} {'p99':>8} {'max':>8}   {'at ceiling':>10}")
for k, nm in enumerate(names):
    c = sample[..., k]
    print(f"  {nm:>10}  {np.median(c):8.0f} {np.percentile(c, 99):8.0f} {c.max():8.0f}"
          f"   {100 * (c >= cfg.ADU_MAX).mean():9.3f}%")
    if np.percentile(c, 99) < cfg.DETECTOR["OFFSET_ADU"] + 20:
        # expected whenever no cell type in the panel is positive for that marker -- a real
        # 40-plex has plenty of these, and they are still legitimate channels
        print(f"{'':14}^ essentially dark: no cell type expresses {nm}")

In [ ]:
def stretch(frame, p=(1.0, 99.5)):
    
    a = np.asarray(frame, np.float32)
    out = np.zeros(a.shape, np.float32)
    for k in range(a.shape[-1]):
        v0, v1 = np.percentile(a[..., k], p)
        out[..., k] = np.clip((a[..., k] - v0) / max(v1 - v0, 1e-6), 0.0, 1.0)
    return out


def as_rgb(frame):
    """First three channels as RGB, with channel 0 (always DAPI) in BLUE by convention."""
    a = stretch(frame)
    rgb = np.zeros(a.shape[:2] + (3,), np.float32)
    for k in range(min(3, a.shape[-1])):
        rgb[..., 2 - k] = a[..., k]        # ch0 -> blue, ch1 -> green, ch2 -> red
    return rgb


nrow, ncol = 3, 4
show = pick[:nrow * ncol]
fig, axes = plt.subplots(nrow, ncol, figsize=(2.7 * ncol, 2.7 * nrow))
for ax, i in zip(np.ravel(axes), show):
    ax.imshow(as_rgb(imgs[i]))            # one frame off disk, per panel
    ax.set_title(f"#{i}", fontsize=8)
    ax.axis("off")
for ax in np.ravel(axes)[len(show):]:
    ax.axis("off")
_lbl = " / ".join(f"{nm}={c}" for nm, c in zip(names[:3], ("blue", "green", "red")))
fig.suptitle(f"{len(show)} of {n} frames   ({_lbl})", fontsize=10)
plt.tight_layout()
plt.show()

# Every channel of a single frame. This is the one to compare against the notebook's own
# tissue render above -- same forward model, so the textures should be recognisably the same.
i0 = int(show[0])
fig, axes = plt.subplots(1, C, figsize=(3.0 * C, 3.4))
for k, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(stretch(imgs[i0])[..., k], cmap="gray")
    ax.set_title(names[k] + (f"  ({dyes[k]})" if dyes else ""), fontsize=9)
    ax.axis("off")
fig.suptitle(f"frame #{i0}, all {C} channels, 1-99.5 percentile stretch", fontsize=10)
plt.tight_layout()
plt.show()